# Technologies

In [1]:
import pandas as pd
import json
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
from itertools import product as iterproduct

# --- Config ---
DATA_PATH = "data/data_tech_swdb.csv"
MIN_VICTIMS = 10        # gruppi con almeno N vittime
MIN_TECH_COUNT = 5      # tech presente in almeno N vittime totali
ALPHA = 0.05
LEVEL = "product"       # "product", "category", o "vendor"

# --- Load & Parse ---
df = pd.read_csv(DATA_PATH)
df["tech_list"] = df["technologies"].apply(json.loads)

# Esplodi tecnologie
rows = []
for _, r in df.iterrows():
    for t in r["tech_list"]:
        rows.append({"account_name": r["account_name"], "group_name": r["group_name"], "tech": t[LEVEL]})
exploded = pd.DataFrame(rows)

# Filtra gruppi piccoli
group_counts = df["group_name"].value_counts()
valid_groups = group_counts[group_counts >= MIN_VICTIMS].index
df_filtered = df[df["group_name"].isin(valid_groups)]
exploded = exploded[exploded["group_name"].isin(valid_groups)]

# Filtra tech rare
tech_victim_counts = exploded.groupby("tech")["account_name"].nunique()
valid_techs = tech_victim_counts[tech_victim_counts >= MIN_TECH_COUNT].index
exploded = exploded[exploded["tech"].isin(valid_techs)]

# --- Matrice presenza (vittima x tech) ---
victim_tech = exploded.groupby(["account_name", "group_name"])["tech"].apply(set).reset_index()
all_techs = sorted(exploded["tech"].unique())

# --- Odds Ratio + Fisher per ogni (gruppo, tech) ---
n_total = df_filtered["account_name"].nunique()
results = []

for grp in valid_groups:
    grp_victims = set(df_filtered[df_filtered["group_name"] == grp]["account_name"])
    other_victims = set(df_filtered[df_filtered["group_name"] != grp]["account_name"])
    n_grp = len(grp_victims)
    n_other = len(other_victims)

    # Set di vittime per tech nel gruppo e fuori
    grp_exploded = exploded[exploded["group_name"] == grp]
    other_exploded = exploded[exploded["group_name"] != grp]
    grp_tech_victims = grp_exploded.groupby("tech")["account_name"].apply(set).to_dict()
    other_tech_victims = other_exploded.groupby("tech")["account_name"].apply(set).to_dict()

    for tech in valid_techs:
        a = len(grp_tech_victims.get(tech, set()))       # gruppo CON tech
        b = n_grp - a                                     # gruppo SENZA tech
        c = len(other_tech_victims.get(tech, set()))      # altri CON tech
        d = n_other - c                                   # altri SENZA tech

        table = [[a, b], [c, d]]
        odds_ratio, p_value = fisher_exact(table, alternative="two-sided")

        results.append({
            "group": grp,
            "tech": tech,
            "a": a, "b": b, "c": c, "d": d,
            "odds_ratio": odds_ratio,
            "p_value": p_value,
            "grp_pct": a / n_grp * 100 if n_grp else 0,
            "other_pct": c / n_other * 100 if n_other else 0,
        })

res = pd.DataFrame(results)

# --- Benjamini-Hochberg correction ---
reject, p_adj, _, _ = multipletests(res["p_value"], alpha=ALPHA, method="fdr_bh")
res["p_adj"] = p_adj
res["significant"] = reject

# --- Output ---
sig = res[res["significant"]].sort_values(["group", "p_adj"])
print(f"\nGruppi analizzati: {len(valid_groups)}")
print(f"Tecnologie analizzate ({LEVEL}): {len(valid_techs)}")
print(f"Test totali: {len(res)}")
print(f"Significativi (BH α={ALPHA}): {len(sig)}\n")

for grp in sig["group"].unique():
    g = sig[sig["group"] == grp].sort_values("odds_ratio", ascending=False)
    print(f"\n{'='*60}")
    print(f"  {grp} (n={int(group_counts[grp])})")
    print(f"{'='*60}")
    over = g[g["odds_ratio"] > 1]
    under = g[g["odds_ratio"] < 1]
    if len(over):
        print(f"\n  OVERREPRESENTED:")
        for _, r in over.head(15).iterrows():
            print(f"    {r['tech']:<40} OR={r['odds_ratio']:>7.2f}  grp={r['grp_pct']:5.1f}% vs {r['other_pct']:5.1f}%  p_adj={r['p_adj']:.4f}")
    if len(under):
        print(f"\n  UNDERREPRESENTED:")
        for _, r in under.head(15).iterrows():
            print(f"    {r['tech']:<40} OR={r['odds_ratio']:>7.2f}  grp={r['grp_pct']:5.1f}% vs {r['other_pct']:5.1f}%  p_adj={r['p_adj']:.4f}")

# Salva tutto
res.to_csv("tech_odds_ratio_results.csv", index=False)
sig.to_csv("tech_odds_ratio_significant.csv", index=False)
print(f"\nSalvati: tech_odds_ratio_results.csv, tech_odds_ratio_significant.csv")


Gruppi analizzati: 65
Tecnologie analizzate (product): 4575
Test totali: 297375
Significativi (BH α=0.05): 4398


  8base (n=94)

  UNDERREPRESENTED:
    Atlassian JIRA                           OR=   0.09  grp=  1.1% vs  10.9%  p_adj=0.0394
    Microsoft System Center                  OR=   0.00  grp=  0.0% vs   8.1%  p_adj=0.0487

  SilentRansomGroup (n=21)

  OVERREPRESENTED:
    Aderant                                  OR= 125.11  grp= 20.0% vs   0.2%  p_adj=0.0001
    Avvo                                     OR=  99.38  grp= 15.0% vs   0.2%  p_adj=0.0019
    Super Lawyers                            OR=  99.38  grp= 15.0% vs   0.2%  p_adj=0.0019
    IntApp                                   OR=  71.52  grp= 10.0% vs   0.2%  p_adj=0.0452

  abyss (n=51)

  OVERREPRESENTED:
    Avaya IP Office                          OR=  12.10  grp=  8.0% vs   0.7%  p_adj=0.0422

  akira (n=227)

  OVERREPRESENTED:
    AfterPay                                 OR=   6.26  grp=  3.1% vs   0.5%  p_adj

# Toufan Group Detailed

In [5]:
# select Toufan Group
import pandas as pd
df = pd.read_csv("data/data_tech_swdb.csv")
df_toufan = df[df["group_name"] == "toufan"]
print(f"Toufan Group vittime: {len(df_toufan)}")

# create file with toufan group attacks
df_toufan.to_csv("output/toufan_attacks.csv", index=False)
print("Salvato: toufan_attacks.csv")


Toufan Group vittime: 33
Salvato: toufan_attacks.csv
